In [ ]:
# ---- Cell 1 - Imports -------------------------------------------------------
import os, time, math, random, warnings
from pathlib import Path
from datetime import datetime
from functools import partial
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from scipy.ndimage import label as scipy_label

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast

torch.backends.cudnn.benchmark = True
import torch._dynamo
torch._dynamo.config.suppress_errors = True

from monai.config import print_config
from monai.utils import set_determinism
from monai.data import CacheDataset, Dataset, DataLoader, pad_list_data_collate
from monai.transforms import (
    Compose, MapTransform, LoadImaged, EnsureChannelFirstd, Orientationd,
    NormalizeIntensityd, CropForegroundd, RandCropByLabelClassesd,
    RandFlipd, RandAffined, RandShiftIntensityd, RandScaleIntensityd,
    RandGaussianNoised, RandGaussianSmoothd, RandAdjustContrastd,
)
from monai.networks.nets import SwinUNETR
from monai.losses import DiceLoss
from monai.metrics import HausdorffDistanceMetric
from monai.inferers import sliding_window_inference
from monai.optimizers import WarmupCosineSchedule

# RandSimulateLowResolutiond requires MONAI >= 1.3
try:
    from monai.transforms import RandSimulateLowResolutiond
    HAS_LOWRES = True
except ImportError:
    HAS_LOWRES = False

# EMA helpers require torch >= 2.0
try:
    from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
    HAS_EMA = True
except ImportError:
    HAS_EMA = False

NUM_GPUS = torch.cuda.device_count()
DEVICE   = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

print_config()
print(f'\nPyTorch {torch.__version__} | GPUs: {NUM_GPUS}')
for i in range(NUM_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  ({p.total_memory/1e9:.1f} GB)')
print(f'Device            : {DEVICE}')
print(f'RandSimulateLowRes: {HAS_LOWRES}')
print(f'EMA available     : {HAS_EMA}')


In [ ]:


DATA_ROOT        = Path(r'/xyz/Data_with_Ground_truth/MICCAI2024-BraTS-GoAT-TrainingData-With-GroundTruth/')
SSL_ENCODER_PATH = Path(r'/xyz/GOAT/Model/model_SSL/ssl_pretrained_encoder.pth')
EXPERIMENTS_ROOT = Path(r'/xyz/Brats2026/GOAT/Model/model_SSL/Fine_tune/')

MODEL_NAME = 'SwinUNETR_SSL_Region'
RUN_DIR    = EXPERIMENTS_ROOT / MODEL_NAME / f'run_{datetime.now():%Y%m%d_%H%M}'
for sub in ['', 'checkpoints', 'figures']:
    (RUN_DIR / sub).mkdir(parents=True, exist_ok=True)
print('Run directory:', RUN_DIR)

SEED = 42
set_determinism(seed=SEED)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

IMAGE_KEYS = ['t1n', 't1c', 't2w', 't2f']
ALL_KEYS   = IMAGE_KEYS + ['label']

# ---- Region-wise training ----------------------------------------------------
# Voxel labels (BraTS 2023+ / GoAT):  1 = NCR, 2 = ED, 3 = ET
# Network predicts 3 NESTED, OVERLAPPING regions with independent sigmoids:
#   channel 0 = TC = {1, 3}
#   channel 1 = WT = {1, 2, 3}
#   channel 2 = ET = {3}
REGION_NAMES = ['TC', 'WT', 'ET']
IN_CHANNELS  = 4
OUT_CHANNELS = 3          # 3-region sigmoid output (TC/WT/ET)
NUM_VOXEL_CLASSES = 4     # bg, NCR, ED, ET - used by RandCropByLabelClassesd

FEATURE_SIZE = 48         # must match the SSL-pretrained encoder

PATCH_SIZE = (96, 96, 96)
SW_ROI     = (96, 96, 96)
SW_BATCH   = 4
SW_OVERLAP = 0.5          # 0.5 during training-time val; 0.7 for final eval
SW_MODE    = 'gaussian'
SW_OVERLAP_FINAL = 0.7

# Class-aware cropping. ratios index = [bg, NCR, ED, ET].
# ET is the bottleneck class, so it gets the largest share.
NUM_CROP_SAMPLES = 4
CROP_CLASS_RATIOS = [1, 2, 1, 3]

MAX_EPOCHS   = 300
VAL_INTERVAL = 2
BATCH_SIZE   = 2 * max(NUM_GPUS, 1)   # x NUM_CROP_SAMPLES patches per step

# ---- Ablation switches. Defaults = Run A. Change ONE per run. ---------------
LEARNING_RATE_ENC    = 1e-5
LEARNING_RATE_DEC    = 1e-4
WEIGHT_DECAY         = 1e-5
WARMUP_EPOCHS        = 0
EXCLUDE_NORM_BIAS_WD = False
AUG_LEVEL            = 'light'
USE_EMA              = False
EMA_DECAY            = 0.999

USE_AMP             = True
AMP_DTYPE           = torch.float16
USE_CHECKPOINTING   = True
EARLY_STOP_PATIENCE = 40              # in VAL rounds, i.e. 80 epochs
DL_NUM_WORKERS      = 8
CACHE_NUM_WORKERS   = 4
VAL_FRACTION        = 0.15
CACHE_RATE_TRAIN    = 1.0

# ---- Post-processing (applied to REGION masks, not voxel classes) -----------
CC_MIN_VOXELS      = {'TC': 50, 'WT': 100, 'ET': 30}
ENFORCE_NESTING    = True   # ET subset of TC subset of WT
ET_DROP_THRESHOLD  = 0      # 0 = DISABLED.

BEST_MODEL_NAME = 'best_swinunetr_ssl_region.pth'
EMA_MODEL_NAME  = 'ema_swinunetr_ssl_region.pth'

print(f'DATA_ROOT        : {DATA_ROOT}')
print(f'SSL_ENCODER_PATH : {SSL_ENCODER_PATH}  exists={SSL_ENCODER_PATH.exists()}')
print(f'Regions          : {REGION_NAMES}  (sigmoid, out_channels={OUT_CHANNELS})')
print(f'Enc LR / Dec LR  : {LEARNING_RATE_ENC} / {LEARNING_RATE_DEC}  warmup={WARMUP_EPOCHS}ep')
print(f'Crop ratios      : bg/NCR/ED/ET = {CROP_CLASS_RATIOS}')
print(f'Aug level        : {AUG_LEVEL}   EMA={USE_EMA}   WD-exclude={EXCLUDE_NORM_BIAS_WD}')
print(f'Sliding window   : roi={SW_ROI} mode={SW_MODE} overlap={SW_OVERLAP} (final {SW_OVERLAP_FINAL})')


In [ ]:
# ---- Cell 3 - Data Discovery + label sanity check ---------------------------
def find_modality(case_dir, key):
    hits = sorted(case_dir.glob(f'*-{key}.nii.gz')) or sorted(case_dir.glob(f'*{key}*.nii*'))
    return str(hits[0]) if hits else None


data_dicts, skipped = [], []

for d in sorted(p for p in DATA_ROOT.iterdir() if p.is_dir()):
    sid  = d.name
    mods = {k: find_modality(d, k) for k in IMAGE_KEYS}
    miss = [k for k, v in mods.items() if v is None]
    if miss:
        skipped.append((sid, f'missing: {miss}')); continue
    seg = sorted(d.glob('*-seg.nii.gz')) or sorted(d.glob('*seg*.nii*'))
    if not seg:
        skipped.append((sid, 'no seg')); continue
    data_dicts.append({**mods, 'label': str(seg[0]), 'subject_id': sid})

print(f'Total labeled cases : {len(data_dicts)}')
print(f'Skipped             : {len(skipped)}')

# ---- Verify the label convention --------------------------------------------
# BraTS 2023+/GoAT uses 1=NCR, 2=ED, 3=ET. If ANY case contains a 4, the older

print('\nLabel sanity check (30 random cases):')
rng_chk   = np.random.default_rng(0)
chk_idx   = rng_chk.choice(len(data_dicts), size=min(30, len(data_dicts)), replace=False)
seen_vals = set()
for i in chk_idx:
    rec  = data_dicts[int(i)]
    vals = np.unique(np.asarray(nib.load(rec['label']).dataobj)).astype(int).tolist()
    seen_vals.update(vals)
print(f'  union of unique label values : {sorted(seen_vals)}')
if 4 in seen_vals:
    raise ValueError('Label 4 found -> old BraTS convention. Region defs must use '
                     'TC={1,4}, WT={1,2,4}, ET={4}. Fix ConvertToRegionsd in Cell 5.')
if not seen_vals.issubset({0, 1, 2, 3}):
    raise ValueError(f'Unexpected label values {sorted(seen_vals)}')
print('  OK: labels are within {0,1,2,3}. Region defs TC={1,3} WT={1,2,3} ET={3} are valid.')


In [ ]:
# ---- Cell 4 - Train/Val Split ------------------------------------------------

rng = np.random.default_rng(SEED)

idx = np.arange(len(data_dicts))
rng.shuffle(idx)
n_val = max(1, int(round(len(idx) * VAL_FRACTION)))
val_idx   = idx[:n_val].tolist()
train_idx = idx[n_val:].tolist()

train_files = [data_dicts[i] for i in train_idx]
val_files   = [data_dicts[i] for i in val_idx]

splits_df = pd.DataFrame(
    [{'subject_id': d['subject_id'], 'split': s}
     for files, s in [(train_files, 'train'), (val_files, 'val')] for d in files]
).sort_values('subject_id').reset_index(drop=True)
splits_df.to_csv(RUN_DIR / 'splits.csv', index=False)

print(f'Train : {len(train_files)}   Val : {len(val_files)}')
print(f'Split saved -> {RUN_DIR / "splits.csv"}')


In [ ]:
# ---- Cell 5 - Transforms ----------------------------------------------------

class ConvertToRegionsd(MapTransform):

    def __call__(self, data):
        d = dict(data)
        for key in self.key_iterator(d):
            l = d[key]
            if isinstance(l, np.ndarray):
                l = torch.as_tensor(l)
            tc = (l == 1) | (l == 3)
            wt = (l == 1) | (l == 2) | (l == 3)
            et = (l == 3)
            d[key] = torch.cat([tc, wt, et], dim=0).float()
        return d


_pre = [
    LoadImaged(keys=ALL_KEYS),
    EnsureChannelFirstd(keys=ALL_KEYS),
    Orientationd(keys=ALL_KEYS, axcodes='RAS'),
    CropForegroundd(keys=ALL_KEYS, source_key='t1c', allow_smaller=True,
                    k_divisible=list(PATCH_SIZE)),
    NormalizeIntensityd(keys=IMAGE_KEYS, nonzero=True, channel_wise=True),
]

# ---- TIER 1: class-aware cropping on the INTEGER label ----------------------

_crop = [
    RandCropByLabelClassesd(
        keys=ALL_KEYS, label_key='label',
        spatial_size=PATCH_SIZE,
        ratios=CROP_CLASS_RATIOS,
        num_classes=NUM_VOXEL_CLASSES,
        num_samples=NUM_CROP_SAMPLES,
        allow_smaller=True,
    ),
]

# Spatial augs must run while the label is still an integer map.

if AUG_LEVEL == 'nnunet':
    _spatial = [
        RandAffined(
            keys=ALL_KEYS, prob=0.3,
            rotate_range=(0.26, 0.26, 0.26),   # +/- 15 degrees
            scale_range=(0.15, 0.15, 0.15),
            mode=('bilinear',) * len(IMAGE_KEYS) + ('nearest',),
            padding_mode='zeros',
        ),
        RandFlipd(keys=ALL_KEYS, spatial_axis=[0], prob=0.5),
        RandFlipd(keys=ALL_KEYS, spatial_axis=[1], prob=0.5),
        RandFlipd(keys=ALL_KEYS, spatial_axis=[2], prob=0.5),
    ]
else:
    _spatial = [
        RandFlipd(keys=ALL_KEYS, spatial_axis=[0], prob=0.5),
        RandFlipd(keys=ALL_KEYS, spatial_axis=[1], prob=0.5),
        RandFlipd(keys=ALL_KEYS, spatial_axis=[2], prob=0.5),
    ]

# Intensity augs only touch IMAGE_KEYS, so they can run after region conversion.
if AUG_LEVEL == 'nnunet':
    _intensity = [
        RandGaussianNoised(keys=IMAGE_KEYS, prob=0.2, std=0.1),
        RandGaussianSmoothd(keys=IMAGE_KEYS, prob=0.15,
                            sigma_x=(0.5, 1.15), sigma_y=(0.5, 1.15), sigma_z=(0.5, 1.15)),
        RandScaleIntensityd(keys=IMAGE_KEYS, factors=0.25, prob=0.3),
        RandShiftIntensityd(keys=IMAGE_KEYS, offsets=0.15, prob=0.3),
        RandAdjustContrastd(keys=IMAGE_KEYS, prob=0.15, gamma=(0.7, 1.5)),
    ]
    if HAS_LOWRES:
        _intensity.append(
            RandSimulateLowResolutiond(keys=IMAGE_KEYS, prob=0.25, zoom_range=(0.5, 1.0))
        )
else:
    _intensity = [
        RandScaleIntensityd(keys=IMAGE_KEYS, factors=0.10, prob=0.5),
        RandShiftIntensityd(keys=IMAGE_KEYS, offsets=0.10, prob=0.5),
        # was std=0.01, which is a no-op on z-scored data
        RandGaussianNoised(keys=IMAGE_KEYS, prob=0.15, std=0.05),
    ]

train_transforms = Compose(_pre + _crop + _spatial +
                           [ConvertToRegionsd(keys=['label'])] + _intensity)

val_transforms = Compose([
    LoadImaged(keys=ALL_KEYS),
    EnsureChannelFirstd(keys=ALL_KEYS),
    Orientationd(keys=ALL_KEYS, axcodes='RAS'),
    CropForegroundd(keys=ALL_KEYS, source_key='t1c', allow_smaller=True),
    NormalizeIntensityd(keys=IMAGE_KEYS, nonzero=True, channel_wise=True),
    ConvertToRegionsd(keys=['label']),
])

print(f'Transforms defined.  AUG_LEVEL={AUG_LEVEL}  low-res sim={HAS_LOWRES}')
print(f'  train: {len(train_transforms.transforms)} transforms')
print(f'  val  : {len(val_transforms.transforms)} transforms')

In [ ]:
# ---- Cell 6 - Datasets & DataLoaders ----------------------------------------


train_ds = CacheDataset(data=train_files, transform=train_transforms,
                        cache_rate=CACHE_RATE_TRAIN, num_workers=CACHE_NUM_WORKERS,
                        progress=True)
val_ds   = CacheDataset(data=val_files, transform=val_transforms,
                        cache_rate=1.0, num_workers=CACHE_NUM_WORKERS,
                        progress=True)

# BATCH_SIZE * NUM_CROP_SAMPLES.
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=DL_NUM_WORKERS, pin_memory=torch.cuda.is_available(),
                          persistent_workers=DL_NUM_WORKERS > 0)
val_loader   = DataLoader(val_ds, batch_size=1, shuffle=False,
                          num_workers=DL_NUM_WORKERS, pin_memory=torch.cuda.is_available(),
                          persistent_workers=DL_NUM_WORKERS > 0,
                          collate_fn=pad_list_data_collate)

print(f'Train dataset : {len(train_ds)} volumes')
print(f'Val   dataset : {len(val_ds)} volumes')
print(f'Train batches : {len(train_loader)}  ({BATCH_SIZE} vols x {NUM_CROP_SAMPLES} patches = '
      f'{BATCH_SIZE * NUM_CROP_SAMPLES} patches/step)')
print(f'Val   batches : {len(val_loader)}')

# quick shape check on one batch
_b = next(iter(train_loader))
print(f'\nSanity: image {_b["t1c"].shape}  label {_b["label"].shape} '
      f'(expect label channels = 3 for TC/WT/ET)  unique={torch.unique(_b["label"]).tolist()}')


In [ ]:
# ---- Cell 7 - Model + SSL Encoder + Optimiser + Loss ------------------------

def _get_raw(m):
    # Unwrap DataParallel and/or torch.compile. Check `module` first so that
    # DataParallel(compile(model)) unwraps in the right order.
    if hasattr(m, 'module'):    m = m.module
    if hasattr(m, '_orig_mod'): m = m._orig_mod
    return m


_base_model = SwinUNETR(
    in_channels    = IN_CHANNELS,
    out_channels   = OUT_CHANNELS,      # 3 regions, not 4 classes
    feature_size   = FEATURE_SIZE,
    use_checkpoint = USE_CHECKPOINTING,
    spatial_dims   = 3,
).to(DEVICE)

# ---- Robust SSL encoder loading ---------------------------------------------

if SSL_ENCODER_PATH.exists():
    enc_state_raw = torch.load(str(SSL_ENCODER_PATH), map_location='cpu')
    if isinstance(enc_state_raw, dict) and 'state_dict' in enc_state_raw:
        enc_state_raw = enc_state_raw['state_dict']

    def strip_prefix(sd, prefix):
        return {k[len(prefix):]: v for k, v in sd.items() if k.startswith(prefix)}

    first_key = next(iter(enc_state_raw))
    if first_key.startswith('module.swinViT.'):
        enc_state = strip_prefix(enc_state_raw, 'module.swinViT.')
        print('DataParallel full-model checkpoint -> stripped module.swinViT.')
    elif first_key.startswith('swinViT.'):
        enc_state = strip_prefix(enc_state_raw, 'swinViT.')
        print('Full-model checkpoint -> stripped swinViT.')
    elif first_key.startswith('module.'):
        enc_state = strip_prefix(enc_state_raw, 'module.')
        print('DataParallel checkpoint -> stripped module.')
    else:
        enc_state = enc_state_raw
        print('Raw swinViT state dict - no prefix stripping needed')

    try:
        missing, unexpected = _base_model.swinViT.load_state_dict(enc_state, strict=True)
        print(f'SSL encoder loaded (strict=True)  missing={len(missing)} unexpected={len(unexpected)}')
    except RuntimeError as e:
        print(f'strict=True failed ({e.__class__.__name__}: {str(e)[:120]})')
        missing, unexpected = _base_model.swinViT.load_state_dict(enc_state, strict=False)
        print(f'SSL encoder loaded (strict=False) missing={len(missing)} unexpected={len(unexpected)}')
        if missing:    print('  Missing (first 5)   :', missing[:5])
        if unexpected: print('  Unexpected (first 5):', unexpected[:5])
    print('Encoder = SSL pretrained | Decoder = random init')
else:
    print(f'WARNING: SSL encoder not found at {SSL_ENCODER_PATH} - random init')

# ---- Param groups from the UNWRAPPED model, BEFORE DataParallel -------------
# TIER 2 (optional): decaying LayerNorm gains and biases hurts transformers.
def split_params(named, wd_exclude):
    decay, no_decay = [], []
    for n, p in named:
        if not p.requires_grad:
            continue
        if wd_exclude and (p.ndim <= 1 or n.endswith('.bias')):
            no_decay.append(p)
        else:
            decay.append(p)
    return decay, no_decay


enc_named = [(n, p) for n, p in _base_model.named_parameters() if n.startswith('swinViT.')]
dec_named = [(n, p) for n, p in _base_model.named_parameters() if not n.startswith('swinViT.')]

enc_d, enc_nd = split_params(enc_named, EXCLUDE_NORM_BIAS_WD)
dec_d, dec_nd = split_params(dec_named, EXCLUDE_NORM_BIAS_WD)

param_groups = [
    {'params': enc_d,  'lr': LEARNING_RATE_ENC, 'weight_decay': WEIGHT_DECAY},
    {'params': dec_d,  'lr': LEARNING_RATE_DEC, 'weight_decay': WEIGHT_DECAY},
]
if EXCLUDE_NORM_BIAS_WD:
    param_groups += [
        {'params': enc_nd, 'lr': LEARNING_RATE_ENC, 'weight_decay': 0.0},
        {'params': dec_nd, 'lr': LEARNING_RATE_DEC, 'weight_decay': 0.0},
    ]

print(f'Encoder params: {sum(p.numel() for _, p in enc_named)/1e6:.2f}M  lr={LEARNING_RATE_ENC}')
print(f'Decoder params: {sum(p.numel() for _, p in dec_named)/1e6:.2f}M  lr={LEARNING_RATE_DEC}')
print(f'Param groups  : {len(param_groups)}  (WD excluded from norm/bias: {EXCLUDE_NORM_BIAS_WD})')

# ---- Wrap AFTER extracting param groups -------------------------------------
if NUM_GPUS > 1:
    model = nn.DataParallel(_base_model)
    print(f'DataParallel across {NUM_GPUS} GPUs')
    print('  NOTE: DataParallel gives ~1.2x on 2 GPUs and serialises the loss on GPU 0.')
    print('        Porting to DDP via torchrun is worth ~1.8x. Not done here to keep')
    print('        this runnable as a notebook.')
else:
    model = _base_model
    print('Single GPU mode')
print(f'Total parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f} M')

# ---- TIER 1: region loss ----------------------------------------------------

loss_function = DiceLoss(
    sigmoid            = True,
    include_background = True,
    batch              = True,
    squared_pred       = True,
    smooth_nr          = 0.0,
    smooth_dr          = 1e-5,
)
print(f'Loss: DiceLoss(sigmoid, batch=True, squared_pred=True) over {REGION_NAMES}')

# ---- Optimiser + schedule ---------------------------------------------------
try:
    optimizer = torch.optim.AdamW(param_groups, fused=True)
    print('Fused AdamW')
except (TypeError, RuntimeError):
    optimizer = torch.optim.AdamW(param_groups)
    print('Standard AdamW')

# TIER 2: WarmupCosineSchedule is a LambdaLR, so it scales each group's own

scheduler = WarmupCosineSchedule(optimizer, warmup_steps=WARMUP_EPOCHS, t_total=MAX_EPOCHS)
print(f'Schedule: {WARMUP_EPOCHS}ep linear warmup -> cosine over {MAX_EPOCHS}ep')

# ---- TIER 3 (optional): weight EMA ------------------------------------------
ema_model = None
if USE_EMA:
    if not HAS_EMA:
        raise RuntimeError('USE_EMA=True but torch.optim.swa_utils.get_ema_multi_avg_fn '
                           'is unavailable. Requires torch >= 2.0.')
    ema_model = AveragedModel(_base_model, multi_avg_fn=get_ema_multi_avg_fn(EMA_DECAY))
    # SwinUNETR uses InstanceNorm/LayerNorm (no running stats), so no update_bn needed.
    print(f'EMA enabled, decay={EMA_DECAY}')

# ---- Inferers built AFTER wrapping ------------------------------------------
def make_inferer(net, overlap=SW_OVERLAP):
    return partial(sliding_window_inference, roi_size=SW_ROI, sw_batch_size=SW_BATCH,
                   predictor=net, overlap=overlap, mode=SW_MODE)


model_inferer = make_inferer(model)

if USE_EMA:
    ema_eval_net = nn.DataParallel(ema_model.module) if NUM_GPUS > 1 else ema_model.module
    ema_inferer  = make_inferer(ema_eval_net)

print('Model ready.')

In [ ]:
# ---- Cell 8 - Metrics + Post-Processing + TTA -------------------------------


def brats_dice(pred_bin, gt_bin):
    # pred_bin, gt_bin: binary tensors/arrays of identical shape, single region
    p = torch.as_tensor(pred_bin).bool()
    g = torch.as_tensor(gt_bin).bool()
    p_sum = int(p.sum()); g_sum = int(g.sum())
    if g_sum == 0:
        return 1.0 if p_sum == 0 else 0.0     # BraTS convention
    inter = int((p & g).sum())
    return 2.0 * inter / (p_sum + g_sum)


def brats_dice_all(pred_regions, gt_regions):
    # (3, H, W, D) binary -> dict over TC/WT/ET
    return {r: brats_dice(pred_regions[i], gt_regions[i]) for i, r in enumerate(REGION_NAMES)}


hd95_metric = HausdorffDistanceMetric(include_background=True, percentile=95.0)

def safe_hd95(pred_regions, gt_regions):
    # Diagnostic only, NOT the selection criterion. Skips degenerate cases rather
    # than applying the BraTS 373.13mm penalty, so it is not leaderboard-exact.
    vals = []
    for i in range(len(REGION_NAMES)):
        p = torch.as_tensor(pred_regions[i]).float()[None, None]
        g = torch.as_tensor(gt_regions[i]).float()[None, None]
        if p.sum() == 0 or g.sum() == 0:
            continue
        try:
            v = float(hd95_metric(y_pred=p, y=g).item())
            hd95_metric.reset()
            if math.isfinite(v):
                vals.append(v)
        except Exception:
            hd95_metric.reset()
    return float(np.mean(vals)) if vals else float('nan')


# ---- Post-processing, on REGIONS ---------------------------------------------
def _remove_small_cc(mask_np, min_voxels):
    if min_voxels <= 0 or mask_np.sum() == 0:
        return mask_np
    out = mask_np.copy()
    labeled, n_comp = scipy_label(out)
    if n_comp == 0:
        return out
    sizes = np.bincount(labeled.ravel())
    for comp_id in range(1, n_comp + 1):
        if sizes[comp_id] < min_voxels:
            out[labeled == comp_id] = 0
    return out


def postprocess_regions(seg):
    # seg: (3, H, W, D) uint8 binary, ordered [TC, WT, ET]
    out = seg.copy()
    for i, r in enumerate(REGION_NAMES):
        out[i] = _remove_small_cc(out[i], CC_MIN_VOXELS[r])

    # Independent sigmoids can violate the nesting the BraTS regions require.
    if ENFORCE_NESTING:
        out[0] = np.maximum(out[0], out[2])   # TC must contain ET
        out[1] = np.maximum(out[1], out[0])   # WT must contain TC

    # DISABLED by default (ET_DROP_THRESHOLD = 0). The old blanket ET->NCR rule at
    # 50 voxels is a BraTS-2020 GLI heuristic. On GoAT, MEN and MET have very
    # different ET statistics, so tune this per tumour type on val or leave it off.
    if ET_DROP_THRESHOLD > 0 and out[2].sum() < ET_DROP_THRESHOLD:
        out[2][:] = 0
        if ENFORCE_NESTING:
            out[0] = np.maximum(out[0], out[2])
    return out


def regions_to_labels(seg):
    # (3, H, W, D) binary [TC, WT, ET] -> integer map 1=NCR, 2=ED, 3=ET.
    # Hierarchical overwrite, for writing NIfTI submissions.
    out = np.zeros(seg.shape[1:], dtype=np.uint8)
    out[seg[1] > 0] = 2
    out[seg[0] > 0] = 1
    out[seg[2] > 0] = 3
    return out


# ---- 8-flip TTA --------------------------------------------------------------
# Replaces the old gamma + 7-degree-rotation views. Gamma is not a symmetry of
# z-scored MRI, and grid_sample rotation introduces interpolation blur exactly at
# the ET boundary you are trying to resolve. Flips are exact and invertible.
FLIP_AXES = [(), (2,), (3,), (4,), (2, 3), (2, 4), (3, 4), (2, 3, 4)]

def tta_inference(net, image, device, overlap=SW_OVERLAP_FINAL):
    probs_sum = None
    net.eval()
    with torch.no_grad():
        for ax in FLIP_AXES:
            xi = torch.flip(image, ax) if ax else image
            xi = xi.to(device)
            with autocast('cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
                logits = sliding_window_inference(
                    xi, roi_size=SW_ROI, sw_batch_size=SW_BATCH,
                    predictor=net, overlap=overlap, mode=SW_MODE,
                )
            p = torch.sigmoid(logits.float()).cpu()
            if ax:
                p = torch.flip(p, ax)
            probs_sum = p if probs_sum is None else probs_sum + p
    return probs_sum / len(FLIP_AXES)


print('Metrics: BraTS-convention region Dice (empty GT + empty pred = 1.0, empty GT + FP = 0.0)')
print(f'Post   : CC {CC_MIN_VOXELS}  nesting={ENFORCE_NESTING}  ET-drop={ET_DROP_THRESHOLD or "off"}')
print(f'TTA    : {len(FLIP_AXES)}-view flip, sliding window mode={SW_MODE}')

In [ ]:
# ---- Cell 9 - Training Loop -------------------------------------------------

scaler = GradScaler('cuda', enabled=USE_AMP and AMP_DTYPE == torch.float16
                                   and torch.cuda.is_available())

best_metric, best_metric_epoch = -1.0, -1
epochs_since_improvement = 0
epoch_loss_values, val_metric_values, log_rows = [], [], []
run_start = time.time()

epoch_bar    = widgets.IntProgress(value=0, min=0, max=MAX_EPOCHS, description='Epochs:',
                                   bar_style='info', layout=widgets.Layout(width='80%'))
status_label = widgets.Label(value='Starting...')
val_label    = widgets.Label(value='')
display(widgets.VBox([epoch_bar, status_label, val_label]))


def run_validation(inferer, tag):
    per_case = []
    with torch.no_grad():
        for vb in val_loader:
            vi = torch.cat([vb[k].to(DEVICE, non_blocking=True) for k in IMAGE_KEYS], dim=1)
            vl = vb['label']                                  # (1, 3, H, W, D) float binary
            with autocast('cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
                vo = inferer(vi)
            pred = (torch.sigmoid(vo.float()) > 0.5).cpu()[0]  # (3, H, W, D)
            gt   = (vl > 0.5)[0]
            row  = brats_dice_all(pred, gt)
            row['HD95'] = safe_hd95(pred, gt)
            per_case.append(row)
    df = pd.DataFrame(per_case)
    return {r: float(df[r].mean()) for r in REGION_NAMES} | {'HD95': float(df['HD95'].mean())}, df


for epoch in range(1, MAX_EPOCHS + 1):

    # ---- TRAIN --------------------------------------------------------------
    model.train()
    epoch_loss, n_steps, epoch_start = 0.0, 0, time.time()

    for batch_data in train_loader:
        inputs = torch.cat([batch_data[k].to(DEVICE, non_blocking=True) for k in IMAGE_KEYS], dim=1)
        labels = batch_data['label'].to(DEVICE, non_blocking=True)   # (B, 3, ...) binary

        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
            outputs = model(inputs)
            loss    = loss_function(outputs, labels)

        if scaler.is_enabled():
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)                       # unscale BEFORE clipping
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        if USE_EMA:
            ema_model.update_parameters(_base_model)

        epoch_loss += loss.item()
        n_steps    += 1

    epoch_loss /= max(n_steps, 1)
    epoch_loss_values.append(epoch_loss)
    scheduler.step()

    enc_lr, dec_lr = optimizer.param_groups[0]['lr'], optimizer.param_groups[1]['lr']
    epoch_bar.value = epoch
    status_label.value = (f'Ep {epoch}/{MAX_EPOCHS}  loss={epoch_loss:.4f}  '
                          f'enc_lr={enc_lr:.2e}  dec_lr={dec_lr:.2e}  '
                          f'{time.time() - epoch_start:.1f}s')

    log_row = {'epoch': epoch, 'train_loss': epoch_loss, 'enc_lr': enc_lr, 'dec_lr': dec_lr,
               'val_TC': float('nan'), 'val_WT': float('nan'), 'val_ET': float('nan'),
               'val_mean': float('nan'), 'val_HD95': float('nan'),
               'epoch_s': round(time.time() - epoch_start, 1)}

    # ---- VALIDATE -----------------------------------------------------------
    if epoch % VAL_INTERVAL == 0:
        model.eval()
        val_start = time.time()

        agg, _ = run_validation(ema_inferer if USE_EMA else model_inferer,
                                'ema' if USE_EMA else 'raw')

        # Selection metric is the mean of the THREE REGION Dices.
        val_mean = (agg['TC'] + agg['WT'] + agg['ET']) / 3.0

        val_metric_values.append({'epoch': epoch, **agg, 'mean': val_mean})
        log_row.update({'val_TC': agg['TC'], 'val_WT': agg['WT'], 'val_ET': agg['ET'],
                        'val_mean': val_mean, 'val_HD95': agg['HD95']})

        improved = ''
        if val_mean > best_metric:
            best_metric, best_metric_epoch = val_mean, epoch
            epochs_since_improvement = 0
            raw = _get_raw(model)
            torch.save(raw.state_dict(), RUN_DIR / BEST_MODEL_NAME)
            torch.save(raw.state_dict(),
                       RUN_DIR / 'checkpoints' / f'ep{epoch:03d}_mean{val_mean:.4f}.pth')
            if USE_EMA:
                torch.save(ema_model.module.state_dict(), RUN_DIR / EMA_MODEL_NAME)
            improved = '  *** new best ***'
            epoch_bar.bar_style = 'success'
        else:
            epochs_since_improvement += 1

        val_label.value = (f'Val ep{epoch}  TC={agg["TC"]:.4f}  WT={agg["WT"]:.4f}  '
                           f'ET={agg["ET"]:.4f}  mean={val_mean:.4f}  '
                           f'HD95={agg["HD95"]:.1f}mm  '
                           f'best={best_metric:.4f}@ep{best_metric_epoch}  '
                           f'{time.time() - val_start:.1f}s{improved}')

        if epochs_since_improvement >= EARLY_STOP_PATIENCE:
            status_label.value = (f'Early stop ep{epoch} - no improvement for '
                                  f'{EARLY_STOP_PATIENCE} val rounds')
            epoch_bar.bar_style = 'warning'
            log_rows.append(log_row)
            pd.DataFrame(log_rows).to_csv(RUN_DIR / 'training_log.csv', index=False)
            break

    log_rows.append(log_row)
    pd.DataFrame(log_rows).to_csv(RUN_DIR / 'training_log.csv', index=False)

elapsed = (time.time() - run_start) / 60
status_label.value = f'Done - {elapsed:.1f} min  |  best mean Dice={best_metric:.4f} @ ep{best_metric_epoch}'
epoch_bar.bar_style = 'success'


In [ ]:
# ---- Cell 10 - Training Curves ----------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(range(1, len(epoch_loss_values) + 1), epoch_loss_values, lw=1.5, color='navy')
axes[0].set(xlabel='Epoch', ylabel='Region DiceLoss', title='Training Loss')
axes[0].grid(alpha=0.3)

if val_metric_values:
    vdf = pd.DataFrame(val_metric_values)
    axes[1].plot(vdf['epoch'], vdf['WT'],   'o-', lw=1.5, color='seagreen',   label='WT')
    axes[1].plot(vdf['epoch'], vdf['TC'],   's-', lw=1.5, color='darkorange', label='TC')
    axes[1].plot(vdf['epoch'], vdf['ET'],   '^-', lw=1.5, color='crimson',    label='ET')
    axes[1].plot(vdf['epoch'], vdf['mean'], '-',  lw=2.5, color='navy', alpha=0.7, label='mean (selection)')
    axes[1].axvline(best_metric_epoch, ls='--', c='navy', alpha=0.4, label=f'best={best_metric:.3f}')
    axes[1].set(xlabel='Epoch', ylabel='Region Dice (BraTS convention)',
                title='Val Dice - TC / WT / ET', ylim=(0, 1))
    axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

    axes[2].plot(vdf['epoch'], vdf['HD95'], 'D-', lw=1.5, color='purple')
    axes[2].set(xlabel='Epoch', ylabel='HD95 (mm)', title='Val HD95 (diagnostic)')
    axes[2].grid(alpha=0.3)

plt.suptitle(f'SwinUNETR SSL Region Fine-Tuning - BraTS GoAT  '
             f'(aug={AUG_LEVEL}, warmup={WARMUP_EPOCHS}, enc_lr={LEARNING_RATE_ENC}, ema={USE_EMA})',
             y=1.03)
plt.tight_layout()
out = RUN_DIR / 'figures' / 'training_curves.png'
plt.savefig(str(out), dpi=140, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

In [ ]:
# ---- Cell 11 - Final Evaluation (8-flip TTA + post-processing) --------------
# Single model. No ensembling.

CKPT = RUN_DIR / (EMA_MODEL_NAME if USE_EMA else BEST_MODEL_NAME)

best_model = SwinUNETR(in_channels=IN_CHANNELS, out_channels=OUT_CHANNELS,
                       feature_size=FEATURE_SIZE, use_checkpoint=False,
                       spatial_dims=3).to(DEVICE)
best_model.load_state_dict(torch.load(str(CKPT), map_location=DEVICE))
best_model.eval()
eval_net = nn.DataParallel(best_model) if NUM_GPUS > 1 else best_model
print(f'Loaded {CKPT.name}  (epoch {best_metric_epoch}, val mean Dice {best_metric:.4f})')

eval_ds     = Dataset(data=val_files, transform=val_transforms)
eval_loader = DataLoader(eval_ds, batch_size=1, shuffle=False, num_workers=4,
                         collate_fn=pad_list_data_collate)

results = []
prog = widgets.IntProgress(value=0, min=0, max=len(eval_loader), description='Eval:',
                           bar_style='info', layout=widgets.Layout(width='75%'))
info = widgets.Label(value='')
display(widgets.VBox([prog, info]))

with torch.no_grad():
    for idx, vb in enumerate(eval_loader, 1):
        sid = vb['subject_id'][0]

        vi = torch.cat([vb[k] for k in IMAGE_KEYS], dim=1)
        gt = (vb['label'] > 0.5)[0].numpy().astype(np.uint8)          # (3, H, W, D)

        probs   = tta_inference(eval_net, vi, DEVICE, overlap=SW_OVERLAP_FINAL)
        raw_seg = (probs[0] > 0.5).numpy().astype(np.uint8)           # (3, H, W, D)
        pp_seg  = postprocess_regions(raw_seg)

        d_raw = brats_dice_all(raw_seg, gt)
        d_pp  = brats_dice_all(pp_seg,  gt)

        results.append({
            'subject_id': sid,
            **{f'Dice_{r}_raw': d_raw[r] for r in REGION_NAMES},
            **{f'Dice_{r}_pp':  d_pp[r]  for r in REGION_NAMES},
            'HD95_pp': safe_hd95(pp_seg, gt),
            'ET_vox_pred': int(pp_seg[2].sum()), 'ET_vox_gt': int(gt[2].sum()),
        })
        prog.value = idx
        info.value = f'[{idx}/{len(eval_loader)}] {sid}  mean_pp=' \
                     f'{np.mean([d_pp[r] for r in REGION_NAMES]):.4f}'

prog.bar_style = 'success'
df = pd.DataFrame(results)
df['Dice_mean_raw'] = df[[f'Dice_{r}_raw' for r in REGION_NAMES]].mean(axis=1)
df['Dice_mean_pp']  = df[[f'Dice_{r}_pp'  for r in REGION_NAMES]].mean(axis=1)
df.to_csv(RUN_DIR / 'val_metrics_tta_pp.csv', index=False)

print('\n' + '=' * 72)
print('  Final Evaluation - single model, 8-flip TTA, BraTS-convention Dice')
print('=' * 72)
for r in REGION_NAMES + ['mean']:
    raw, pp = df[f'Dice_{r}_raw'].mean(), df[f'Dice_{r}_pp'].mean()
    print(f'  {r:<5}  raw={raw:.4f}   +post-proc={pp:.4f}   gain={pp - raw:+.4f}')
print('=' * 72)


In [ ]:
# ---- Cell 12 - Summary ------------------------------------------------------
print('=' * 72)
print(f' Run directory: {RUN_DIR}')
print('=' * 72)
for f in sorted(RUN_DIR.rglob('*')):
    if f.is_file():
        print(f'  {str(f.relative_to(RUN_DIR)):<52}  {f.stat().st_size/1e6:>7.2f} MB')
print('=' * 72)
print(f'Config       : aug={AUG_LEVEL}  warmup={WARMUP_EPOCHS}  enc_lr={LEARNING_RATE_ENC}  '
      f'ema={USE_EMA}  wd_excl={EXCLUDE_NORM_BIAS_WD}')
print(f'Best model   : {RUN_DIR / (EMA_MODEL_NAME if USE_EMA else BEST_MODEL_NAME)}')
print(f'Val mean Dice: {best_metric:.4f} @ epoch {best_metric_epoch}  (selection metric)')
if len(df):
    print(f'Final TTA+pp : TC={df["Dice_TC_pp"].mean():.4f}  WT={df["Dice_WT_pp"].mean():.4f}  '
          f'ET={df["Dice_ET_pp"].mean():.4f}  mean={df["Dice_mean_pp"].mean():.4f}')


## TTA / Post-processing ablation

Runs all four configurations (raw, +TTA, +PP, +TTA+PP) on the internal validation split, using the model already loaded above.

In [ ]:


from scipy.ndimage import binary_erosion

def get_surface(mask_bin):
    """Boolean surface (outermost) voxels of a binary mask."""
    if mask_bin.sum() == 0:
        return np.zeros_like(mask_bin, dtype=bool)
    eroded = binary_erosion(mask_bin, border_value=0)
    return mask_bin & ~eroded


def nsd_metric(pred_bin, gt_bin, tolerance_mm=1.0, spacing=None):
    """
    Normalized Surface Dice at a fixed tolerance. Returns NaN if either mask
    is empty (no surface to measure against).
    """
    spacing = np.array([1.0, 1.0, 1.0]) if spacing is None else np.array(spacing)
    pred_bin = np.asarray(pred_bin).astype(bool)
    gt_bin   = np.asarray(gt_bin).astype(bool)

    if pred_bin.sum() == 0 or gt_bin.sum() == 0:
        return np.nan

    pred_surf = get_surface(pred_bin)
    gt_surf   = get_surface(gt_bin)

    from scipy.ndimage import distance_transform_edt
    pred_dist = distance_transform_edt(~pred_bin, sampling=spacing)
    gt_dist   = distance_transform_edt(~gt_bin,   sampling=spacing)

    # Fraction of each surface within tolerance of the other mask
    close_pred = (gt_dist[pred_surf] <= tolerance_mm).sum() if pred_surf.sum() > 0 else 0
    close_gt   = (pred_dist[gt_surf] <= tolerance_mm).sum() if gt_surf.sum() > 0 else 0
    total = pred_surf.sum() + gt_surf.sum()
    if total == 0:
        return np.nan
    return (close_pred + close_gt) / total


def nsd_all(pred_regions, gt_regions, tolerance_mm=1.0, region_names=REGION_NAMES):
    return {
        r: nsd_metric(pred_regions[i], gt_regions[i], tolerance_mm=tolerance_mm)
        for i, r in enumerate(region_names)
    }


def inference_no_tta(net, image, device, overlap=SW_OVERLAP_FINAL):
    """Single-pass inference, no flip averaging."""
    net.eval()
    with torch.no_grad():
        image = image.to(device)
        with autocast('cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
            logits = sliding_window_inference(
                image, roi_size=SW_ROI, sw_batch_size=SW_BATCH,
                predictor=net, overlap=overlap, mode=SW_MODE,
            )
        return torch.sigmoid(logits.float()).cpu()


def run_ablation_config(use_tta, use_pp, loader, net, device, tag):
    dice_acc = {r: [] for r in REGION_NAMES}
    nsd_acc  = {r: [] for r in REGION_NAMES}
    hd95_acc = {r: [] for r in REGION_NAMES}

    with torch.no_grad():
        for vb in loader:
            vi = torch.cat([vb[k] for k in IMAGE_KEYS], dim=1)
            gt = (vb['label'] > 0.5)[0].numpy().astype(np.uint8)

            if use_tta:
                probs = tta_inference(net, vi, device, overlap=SW_OVERLAP_FINAL)
            else:
                probs = inference_no_tta(net, vi, device, overlap=SW_OVERLAP_FINAL)

            seg = (probs[0] > 0.5).numpy().astype(np.uint8)
            if use_pp:
                seg = postprocess_regions(seg)

            d = brats_dice_all(seg, gt)
            n = nsd_all(seg, gt)
            for i, r in enumerate(REGION_NAMES):
                dice_acc[r].append(d[r])
                if not np.isnan(n[r]):
                    nsd_acc[r].append(n[r])
                p = torch.as_tensor(seg[i]).float()[None, None]
                g = torch.as_tensor(gt[i]).float()[None, None]
                if p.sum() > 0 and g.sum() > 0:
                    try:
                        v = float(hd95_metric(y_pred=p, y=g).item())
                        hd95_metric.reset()
                        if math.isfinite(v):
                            hd95_acc[r].append(v)
                    except Exception:
                        hd95_metric.reset()

    row = {'config': tag}
    for r in REGION_NAMES:
        row[f'DSC_{r}']  = float(np.mean(dice_acc[r])) if dice_acc[r] else float('nan')
        row[f'NSD_{r}']  = float(np.mean(nsd_acc[r]))  if nsd_acc[r]  else float('nan')
        row[f'HD95_{r}'] = float(np.mean(hd95_acc[r])) if hd95_acc[r] else float('nan')
    return row


ablation_loader = DataLoader(Dataset(data=val_files, transform=val_transforms),
                             batch_size=1, shuffle=False, num_workers=4,
                             collate_fn=pad_list_data_collate)

configs = [
    ('raw',    False, False),
    ('tta',    True,  False),
    ('pp',     False, True),
    ('tta_pp', True,  True),
]

ablation_rows = []
for tag, use_tta, use_pp in configs:
    print(f'Running config: {tag}  (TTA={use_tta}, PP={use_pp})')
    ablation_rows.append(
        run_ablation_config(use_tta, use_pp, ablation_loader, eval_net, DEVICE, tag)
    )

ablation_df = pd.DataFrame(ablation_rows)
ablation_df.to_csv(RUN_DIR / 'tta_pp_ablation.csv', index=False)
print('\nSaved:', RUN_DIR / 'tta_pp_ablation.csv')
print(ablation_df.to_string(index=False))
